In [1]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit version:", qiskit.__version__)
print("Qiskit Aer version:", qiskit_aer.__version__)
print("Qiskit IBM Runtime version:", qiskit_ibm_runtime.__version__)

# Build a trivial circuit to confirm things actually work end-to-end
from qiskit import QuantumCircuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()
print(qc.draw())

Qiskit version: 2.5.0
Qiskit Aer version: 0.17.2
Qiskit IBM Runtime version: 0.48.0
        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 


In [5]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

simulator = AerSimulator()

qc = QuantumCircuit(2,2)
qc.rx(np.pi/2,1)
qc.cx(1,0)
print("Statevector:")
display(Statevector(qc).draw('latex'))

qc.measure([1,0],[1,0])

print("Circuit:")
display(qc.draw('text'))

qc_t = transpile(qc, simulator)
counts = simulator.run(qc, shots=2**10).result().get_counts()

print("Probability Distribution:")
def print_distribution(counts):
    total = sum(counts.values())
    for outcome, count in sorted(counts.items()):
        pct = count / total * 100
        bar = '█' * int(pct / 2)  # scale so 100% = 50 chars wide
        print(f"{outcome}: {count:5d} ({pct:5.1f}%) {bar}")

print_distribution(counts)

Statevector:


<IPython.core.display.Latex object>

Circuit:


┌───┐   ┌─┐
q_0: ───────────┤ X ├───┤M├
     ┌─────────┐└─┬─┘┌─┐└╥┘
q_1: ┤ Rx(π/2) ├──■──┤M├─╫─
     └─────────┘     └╥┘ ║ 
c: 2/═════════════════╩══╩═
                      1  0

Probability Distribution:
00:   542 ( 52.9%) ██████████████████████████
11:   482 ( 47.1%) ███████████████████████


In [ ]:
import sys


!{sys.executable} -m pip install matplotlib

In [3]:
print(qc.draw('text'))

                ┌───┐   ┌─┐
q_0: ───────────┤ X ├───┤M├
     ┌─────────┐└─┬─┘┌─┐└╥┘
q_1: ┤ Rx(π/2) ├──■──┤M├─╫─
     └─────────┘     └╥┘ ║ 
c: 2/═════════════════╩══╩═
                      1  0 


In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account('bcdtslwXwAn9JVHIyPAtidGAjDHzRJlZpjB8kGFuSpEL')

This is the start of reversible XOR

In [7]:
def x(a_in):
    a_out = a_in ^ 1   # Remember ^ performs the XOR operation. a^1 = a̅
    return a_out

In [8]:
def cx(a_in, b_in):
    
    # a' is always equal to a.
    a_out = a_in
    
    # if control is one (i.e., a == 1): negate target (b' = b̅), 
    # else: leave b alone (b' = b).
    if a_in == 1:                
        b_out = x(b_in)
    else:
        b_out = b_in

    return (a_out, b_out)

In [9]:
print('CX gate:')
print('a | b || a\' | b\' |')

# Iterate over possible combinations of a and b
for a, b in [(0,0), (0,1), (1,0), (1,1)]:
    ap, bp = cx(a,b)
    print(f'{a} | {b} || {ap}  | {bp}  |')

CX gate:
a | b || a' | b' |
0 | 0 || 0  | 0  |
0 | 1 || 0  | 1  |
1 | 0 || 1  | 1  |
1 | 1 || 1  | 0  |


In [10]:
print('Two CX gates:')
print('a | b || a\'\' | b\'\' |')

# Apply CX twice for all combination of inputs a, b.
for a, b in [(0,0), (0,1), (1,0), (1,1)]:
    ai, bi = cx(a,b)
    ap, bp = cx(ai,bi)
    print(f'{a} | {b} || {ap}   | {bp}   |')

Two CX gates:
a | b || a'' | b'' |
0 | 0 || 0   | 0   |
0 | 1 || 0   | 1   |
1 | 0 || 1   | 0   |
1 | 1 || 1   | 1   |


This is start of reversible copy/ fanning, just use target b as 0

Toffoli gate for AND CCX

2 control, one target with X , here X is when both controls are ON.

When target 0, c' imitates AND             when target = 1, NAND gate

In [22]:
def ccx(a_in, b_in, c_in):
    if (a_in==1 and b_in ==1):
        c_out = x(c_in)
    else:
        c_out = 0
    return (a_in,b_in,c_out)

In [24]:
for a, b in [(0,0), (0,1), (1,0), (1,1)]:
    ap, bp, cp = ccx(a,b,0)
    print(f'{a} | {b} || {ap}  | {bp}  |{cp}  |')

0 | 0 || 0  | 0  |0  |
0 | 1 || 0  | 1  |0  |
1 | 0 || 1  | 0  |0  |
1 | 1 || 1  | 1  |1  |


For or gate, write as negation(a' ^ b'). a and b both have X in control, and c is 1 in target. rest is same as toffoli, for reversability add both circuits in reverse order

Reversible full adder using circuit design

In [25]:
print('Reversible Full Adder')
print(f'a | b | cin || s | cout |')

# Iterate over possible combinations of a, b and cin
inputs = [(0,0,0), (0,0,1), (0,1,0), (0,1,1),
          (1,0,0), (1,0,1), (1,1,0), (1,1,1)]

for a, b, c in inputs:
    a1, b1, d1 = ccx(a, b,0)      # AND1
    a2, b2 = cx(a1, b1)            # XOR1
    a3, c3, e3 = ccx(b2, c,0)     # AND2
    b4, s = cx(b2, c3)             # XOR2
    
    d5 = x(d1)                     # OR ...
    e5 = x(e3)                     # 
    d6, e6, cout = ccx(d5, e5, 1)  #

    print(f'{a} | {b} | {c}   || {s} | {cout}    |')

Reversible Full Adder
a | b | cin || s | cout |
0 | 0 | 0   || 0 | 0    |
0 | 0 | 1   || 1 | 0    |
0 | 1 | 0   || 1 | 0    |
0 | 1 | 1   || 0 | 0    |
1 | 0 | 0   || 1 | 0    |
1 | 0 | 1   || 0 | 0    |
1 | 1 | 0   || 0 | 0    |
1 | 1 | 1   || 1 | 0    |


In [1]:
import numpy as np
import sympy as sp

In [2]:
ket_0 = np.array([[1],
                  [0]])

In [3]:
sp.Matrix(ket_0)

Matrix([
[1],
[0]])

In [29]:
mag_0 = np.sqrt(np.sum(ket_0**2))

In [30]:
mag_0

np.float64(1.0)

In [31]:
dot_prods = np.vdot(ket_0, ket_0)

In [32]:
print(dot_prods)

1


In [5]:
X = np.array([[0,1],
              [1,0]])
sp.Matrix(X)

Matrix([
[0, 1],
[1, 0]])

In [6]:
ket_1 = X @ ket_0
sp.Matrix(ket_1)

Matrix([
[0],
[1]])

changing ket from 0 to 1, single bit gate only eproforms this

property of the matrices that represent reversible gates/circuits: they must be invertible, kroneker product

|1⟩ ⊗ |0⟩ = |10⟩     kronecker product

In [8]:
b = np.kron(ket_1, ket_0)
sp.Matrix(b)

Matrix([
[0],
[0],
[1],
[0]])

to tell that 1 comes where the value is. at the pos corresponding to value of b

In [10]:
def bin_to_vec(b):
    brev = b[::-1]
    for i, bit in enumerate(brev):
        if bit == '0':
            bit_vec = ket_0
        else:
            bit_vec = ket_1

        if i==0 :
            bvec = bit_vec
        else:
            bvec = np.kron(bit_vec,bvec)
    return bvec
        
        

In [12]:
bin_to_vec('100')

array([[0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0]])

In [13]:
b_vec = bin_to_vec(b)                    # Construct vector using bin_to_vec function
b_vec_len = len(b_vec)                   # Find length of vector
b_vec_one = np.where(b_vec == 1)[0][0]   # Find position in vector where entry is equal to 1
